# MABRIG GovFriction Lab — Independent Kaggle Analysis

**BFS-DAI v0.1 | CPU only.** Upload the exact frozen CSV used by the browser app as a Kaggle notebook input. The data is not bundled with this notebook. Outputs are `kaggle_results.csv` and `kaggle_results.manifest.json`.

The same documented method is implemented independently in Python; it does not import browser JavaScript. Agreement is a code consistency check, not proof of construct validity. See `docs/METHODS.md` in the accompanying source package.


In [ ]:
from pathlib import Path
INPUT_FILE = None  # Set explicitly to Path('/kaggle/input/YOUR-DATA/nyc311_dsny_2025_q1.csv') if needed
if INPUT_FILE is None:
    choices = sorted(Path('/kaggle/input').rglob('*.csv'))
    preferred = [p for p in choices if p.name == 'nyc311_dsny_2025_q1.csv']
    if len(preferred)==1:
        INPUT_FILE = preferred[0]
    elif len(choices)==1:
        INPUT_FILE = choices[0]
    else:
        raise FileNotFoundError('Provide INPUT_FILE manually. Found candidates: ' + str(choices[:30]))
print('Frozen dataset:', INPUT_FILE)


## Reference calculations

Closure is based on `closed_date` and the horizon; it is not evidence of physical problem resolution. Input records are only analyzed after the dataset checksum is calculated. The code below is a standalone independent statistical implementation, not an import from the browser app.


In [ ]:
"""Independent Kaggle-compatible reference implementation of BFS-DAI v0.1.

This script deliberately shares NO runtime calculation code with the browser JS app.
Standard library only. Use on identical SHA-256 frozen input CSV as the app.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import math
from collections import Counter, defaultdict
from datetime import date, datetime
from pathlib import Path
from statistics import mean, median

SOURCE_FIELDS = ('unique_key','created_date','closed_date','agency','complaint_type','borough')
BOROUGHS = {'BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND'}
SERVICES = {'Graffiti', 'Missed Collection (All Materials)'}


def local_timestamp(value):
    """Read naive local wall-clock date/time without OS timezone transformation."""
    value = (value or '').strip()
    if not value:
        return None
    try:
        if '/' in value:
            return datetime.strptime(value, '%m/%d/%Y %I:%M:%S %p') if 'AM' in value.upper() or 'PM' in value.upper() else datetime.strptime(value, '%m/%d/%Y %H:%M:%S')
        return datetime.fromisoformat(value.replace('T', ' ').removesuffix('Z'))
    except ValueError:
        return False


def rate_interval(successes, total):
    # Wilson 95% score interval, z = quantile 0.975 of standard normal
    z = 1.959963984540054
    p = successes/total
    z2 = z*z
    middle = (p + z2/(2*total))/(1+z2/total)
    radius = z*math.sqrt(p*(1-p)/total + z2/(4*total*total))/(1+z2/total)
    return max(0, middle-radius), min(1, middle+radius)


def summarize(cases, horizon):
    total = len(cases)
    counts_over = sum(1 for elapsed in cases if elapsed > horizon)
    restricted = sum(min(elapsed, horizon)/horizon for elapsed in cases)
    nonclosure = counts_over/total
    delayed = restricted/total
    observed = [v for v in cases if math.isfinite(v)]
    lo,hi = rate_interval(counts_over, total)
    return dict(n=total, nonclosure30=nonclosure, delayBurden30=delayed,
                friction=(nonclosure+delayed)/2, lateCount=counts_over,
                medianClosedDays=median(observed) if observed else None,
                wilsonLow=lo,wilsonHigh=hi)


def calculate(raw, *, date_start='2025-01-01', date_end='2025-04-01', snapshot='2026-11-19', horizon=30, threshold=10, friction_weight=.70):
    if not 0 <= friction_weight <= 1 or horizon <= 0 or threshold < 1:
        raise ValueError('Invalid parameters')
    date_start = datetime.combine(date.fromisoformat(date_start),datetime.min.time())
    date_end = datetime.combine(date.fromisoformat(date_end),datetime.min.time())
    snapshot = datetime.combine(date.fromisoformat(snapshot),datetime.min.time())
    if date_start >= date_end: raise ValueError('Invalid inclusion window')
    tracked = Counter(input=len(raw))
    uniqueness=set(); eligible=[]
    for entry in raw:
        ident=(entry.get('unique_key') or '').strip()
        created=local_timestamp(entry.get('created_date'))
        if not ident or created is None or created is False:
            tracked['invalidCreated']+=1;continue
        if ident in uniqueness:
            tracked['duplicate']+=1;continue
        uniqueness.add(ident)
        service=(entry.get('complaint_type') or '').strip()
        borough=(entry.get('borough') or '').strip().upper()
        if not(date_start <= created < date_end and (entry.get('agency') or '').strip().upper()=='DSNY' and service in SERVICES):
            tracked['scopeExcluded']+=1;continue
        if borough not in BOROUGHS:
            tracked['invalidLocationService']+=1;continue
        closed_text=(entry.get('closed_date') or '').strip()
        closed=local_timestamp(closed_text)
        if closed is False or (closed is not None and closed < created):
            tracked['invalidClosed']+=1;continue
        if created>snapshot:
            tracked['scopeExcluded']+=1;continue
        if (snapshot-created).total_seconds() < horizon*86400:
            tracked['immature']+=1;continue
        if (entry.get('status') or '').strip().lower()=='closed' and not closed_text:
            tracked['statusClosedWithoutDate']+=1
        elapsed=math.inf if closed is None or closed>snapshot else (closed-created).total_seconds()/86400
        eligible.append((service,borough,elapsed))
    tracked['eligible']=len(eligible)
    category=defaultdict(list)
    for service,borough,elapsed in eligible:
        category[(service,borough)].append(elapsed)
    scored=[]
    for service,borough in sorted(category):
        values=category[(service,borough)]
        peers=[elapsed for (serv,other),vals in category.items() if serv==service and other!=borough for elapsed in vals]
        own=summarize(values,horizon)
        qualified=len(values)>=threshold and len(peers)>=threshold
        other=summarize(peers,horizon) if qualified else None
        asym=abs(own['friction']-other['friction']) if qualified else None
        composite=100*(friction_weight*own['friction']+(1-friction_weight)*asym) if qualified else None
        if len(values)<threshold:
            own={**own,**{key:None for key in ('nonclosure30','delayBurden30','friction','lateCount','medianClosedDays','wilsonLow','wilsonHigh')}}
        scored.append(dict(service=service,borough=borough,**own,peerN=len(peers),
                           peerFriction=other['friction'] if qualified else None,asymmetry=asym,
                           score=composite,warning=('Insufficient comparable records' if not qualified else 'Exploratory: n < 30' if len(values)<30 else '')))
    return tracked, scored


def main():
    parser=argparse.ArgumentParser(description='Run independent BFS-DAI reference against frozen 311 CSV')
    parser.add_argument('input_csv',type=Path)
    parser.add_argument('--out',type=Path,default=Path('kaggle_results.csv'))
    parser.add_argument('--from-date',default='2025-01-01')
    parser.add_argument('--to-date',default='2025-04-01')
    parser.add_argument('--snapshot',default='2026-11-19')
    parser.add_argument('--horizon',type=int,default=30)
    parser.add_argument('--min-n',type=int,default=10)
    parser.add_argument('--weight',type=float,default=.70)
    a=parser.parse_args()
    content=a.input_csv.read_bytes()
    sha=hashlib.sha256(content).hexdigest()
    with a.input_csv.open(newline='',encoding='utf-8-sig') as handle:
        rd=csv.DictReader(handle)
        missing=[c for c in SOURCE_FIELDS if c not in (rd.fieldnames or [])]
        if missing: parser.error('Missing required columns: '+', '.join(missing))
        rows=list(rd)
    counters,groups=calculate(rows,date_start=a.from_date,date_end=a.to_date,snapshot=a.snapshot,
         horizon=a.horizon,threshold=a.min_n,friction_weight=a.weight)
    output=[]
    for x in groups: output.append({**x, 'source_sha256':sha,'protocol':'bfs-dai-0.1'})
    fields=['service','borough','n','nonclosure30','delayBurden30','friction','lateCount',
       'medianClosedDays','wilsonLow','wilsonHigh','peerN','peerFriction','asymmetry','score','warning','source_sha256','protocol']
    a.out.parent.mkdir(parents=True,exist_ok=True)
    with a.out.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(output)
    manifest={'protocol':'bfs-dai-0.1','source_file':a.input_csv.name,'source_sha256':sha,
       'parameters':{'from':a.from_date,'to':a.to_date,'snapshot':a.snapshot,'horizonDays':a.horizon,
                     'minN':a.min_n,'wFriction':a.weight},'counts':dict(counters),'scored_groups':len(groups),
       'limitations':['Records not guaranteed to equal actual service resolution','Independent implementation but not independent external validation','Synthetic results must not be cited as empirical findings']}
    a.out.with_suffix('.manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print(json.dumps({'output':str(a.out),'manifest':str(a.out.with_suffix('.manifest.json')),'counts':dict(counters),
      'groups':len(groups),'SHA256':sha},indent=2))




In [ ]:
import csv, hashlib, json
from datetime import datetime
OUTPUT_CSV=Path('/kaggle/working/kaggle_results.csv')
raw_bytes=INPUT_FILE.read_bytes()
sha256=hashlib.sha256(raw_bytes).hexdigest()
with INPUT_FILE.open(newline='', encoding='utf-8-sig') as fh:
    reader=csv.DictReader(fh)
    missing=[key for key in SOURCE_FIELDS if key not in (reader.fieldnames or [])]
    if missing:
        raise ValueError('Missing source columns: '+str(missing))
    raw_records=list(reader)
counts, groups = calculate(raw_records, date_start='2025-01-01', date_end='2025-04-01', snapshot='2026-11-19', horizon=30, threshold=10, friction_weight=0.70)
columns=['service','borough','n','nonclosure30','delayBurden30','friction','lateCount','medianClosedDays','wilsonLow','wilsonHigh','peerN','peerFriction','asymmetry','score','warning','source_sha256','protocol']
with OUTPUT_CSV.open('w',newline='',encoding='utf-8') as fh:
    writer=csv.DictWriter(fh,fieldnames=columns)
    writer.writeheader()
    for g in groups:
        writer.writerow({**g,'source_sha256':sha256,'protocol':'bfs-dai-0.1'})
manifest={'protocol':'bfs-dai-0.1','source_file':INPUT_FILE.name,'source_sha256':sha256,'counts':dict(counts),'group_count':len(groups),'snapshot':'2026-11-19', 'horizon_days':30,'minimum_n':10,'friction_weight':0.70, 'synthetic': 'synthetic' in INPUT_FILE.name.lower()}
OUTPUT_CSV.with_suffix('.manifest.json').write_text(json.dumps(manifest,indent=2))
print('Source SHA-256:',sha256)
print('Eligibility counts:',dict(counts))
print('Group count:',len(groups))
print('Saved:',OUTPUT_CSV,OUTPUT_CSV.with_suffix('.manifest.json'))


In [ ]:
# A numerical output preview (descriptive only)
for group in groups:
    print(f"{group['service'][:19]:19} {group['borough']:14} n={group['n']:4d} BFS-DAI={group['score'] if group['score'] is not None else 'SUPPRESSED'}")


## Interpretation and next step

Download both output files from the Kaggle `Output` panel. In the browser app, run the **same CSV** with unchanged settings and upload `kaggle_results.csv` into “Check against Kaggle.” A SHA-256 mismatch invalidates comparison. The pilot is not causal; the included source package's synthetic test data are not NYC observations.
